# Train invoice extraction on your reviewed PDFs
Use a **fresh T4 GPU runtime** (Runtime → Change runtime type → T4).
This notebook prepares drafts and trains only after you review their answers.
111 PDFs alone are not verified labels. Training has not already happened.
Run cells in order; pause at review. Keep this runtime separate from OCR/Ollama.


## Step 1: Install project
Start here in a fresh T4 runtime. Wait for **Step 1 complete** before continuing.


In [ ]:
import subprocess, sys, os
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'training/requirements.txt'], check=True)
(PROJECT / '.training-setup-ready').write_text('Dependencies installed successfully\n')
print('Step 1 complete. Run Step 2: Prepare pages next.')


## Step 2: Prepare pages
Run after Step 1 completes successfully.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print(torch.cuda.get_device_name(0))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
from training.data import prepare, export
print(prepare(PROJECT / 'public_invoice_pdfs', WORKSPACE))
print('Step 2 complete. You can now generate suggested answers.')


## Optional: generate suggested answers
First test only `9479.pdf` with `DRAFT_LIMIT = 1`. Do not run the full batch until
this result is checked. Each scope has a 120-second generation budget, checked
between model decoding steps (not a hard process timeout). Header and item limits
are 1024 and 2048 output tokens. Incomplete JSON remains an error.
Draft generation is **not** training.
Saved suggestions never automatically become verified labels.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
DRAFT_LIMIT = 1
DRAFT_PDF = '9479.pdf'  # Use '' for the next unreviewed page.
command = [sys.executable, '-u', '-m', 'training.run', 'draft', '--workspace', str(WORKSPACE), '--limit', str(DRAFT_LIMIT)]
if DRAFT_PDF:
    command += ['--pdf', DRAFT_PDF, '--force']
subprocess.run(command, check=True)


## Review labels
Choose a page and click **Load model draft** if available. Correct the JSON against
the image. Add extra printed labels under `other_fields` and annotations under
`handwritten_notes`. Use the same layout group for every invoice of a supplier/template.
Enter your reviewer name, check the verification box only after checking the whole
page, then **Save review**. Do this across at least three layout groups before
exporting; review the entire collection for the actual experiment.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
from training.review import review
review(WORKSPACE)


## Back up labels before training
Download this ZIP now and again after more review. It contains page images and labels.
To resume later, upload it through Colab's Files sidebar and extract it back into
`/content/OCR-training/training_workspace/review` before rerunning preparation.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
import shutil
from google.colab import files
backup = shutil.make_archive('/content/invoice-review', 'zip', root_dir=WORKSPACE)
files.download(backup)


## Export and inspect split coverage
This cell intentionally stops if there are no verified labels or fewer than three
layout groups. Pages and duplicated documents cannot cross splits. Inspect the
summary and correct any supplier groups before training.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
from training.data import export
print(export(WORKSPACE, PROJECT / 'public_invoice_pdfs'))


## Baseline on held-out pages

In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
subprocess.run([sys.executable, '-u', '-m', 'training.run', 'evaluate', '--workspace', str(WORKSPACE), '--output', 'model_artifacts/base'], check=True)


## Train LoRA
Two epochs by default. Checkpoints remain in the output directory. A Free Colab
session can disconnect: download checkpoints or save them to your own persistent
storage. Too-long samples fail instead of silently losing fields.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
RESUME_CHECKPOINT = ''  # Set to a saved checkpoint directory to resume.
command = [sys.executable, '-u', '-m', 'training.run', 'train', '--workspace', str(WORKSPACE), '--output', str(OUTPUT)]
if RESUME_CHECKPOINT:
    command += ['--resume', RESUME_CHECKPOINT]
subprocess.run(command, check=True)


## Evaluate the trained adapter on the same held-out pages

In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
subprocess.run([sys.executable, '-u', '-m', 'training.run', 'evaluate', '--workspace', str(WORKSPACE), '--adapter', str(OUTPUT / 'adapter'), '--output', 'model_artifacts/tuned'], check=True)
import json
for variant in ['base', 'tuned']:
    result = json.loads((PROJECT / 'model_artifacts' / variant / 'evaluation.json').read_text())
    print(variant, 'field accuracy:', result['field_exact_accuracy'], 'missing fields:', result['missing_field_count'])


## Download model and results
This saves the adapter, checkpoints and evaluation reports. The original Ollama
model remains unchanged; use the evaluator to test the adapter before deployment.


In [ ]:
import os, sys, subprocess
from pathlib import Path
PROJECT = Path('/content/OCR-training')
if not (PROJECT / 'training' / '__init__.py').is_file() or not (PROJECT / '.training-setup-ready').is_file():
    raise RuntimeError('Setup is missing or incomplete. Run Step 1: Install project, then Step 2: Prepare pages at the top of this notebook. Do not pip install training.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
WORKSPACE = PROJECT / 'training_workspace/review'
OUTPUT = PROJECT / 'model_artifacts/invoice-qwen3-vl'
if not (WORKSPACE / 'inventory.json').is_file():
    raise RuntimeError('Pages are not prepared. Run Step 2: Prepare pages before this cell.')
import shutil
from google.colab import files
artifact = shutil.make_archive('/content/invoice-model-results', 'zip', root_dir=PROJECT / 'model_artifacts')
files.download(artifact)
